In [ ]:
# src/data/dataset.py
import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from typing import List, Callable, Optional, Tuple

class PlantDataset(Dataset):
    """
    Custom PyTorch Dataset for loading plant disease images and labels 
    from explicit paths and label lists.
    """
    def __init__(
        self, 
        image_paths: List[str], 
        labels: List[int], 
        transform: Optional[Callable] = None
    ):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise IOError(f"Failed to load image at {img_path}: {e}")

        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader

# 1. Define standard preprocessing transforms for EfficientNet-B0
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    )
])

# 2. Point to your actual image paths and corresponding integer labels
# (Replace these placeholders with your real file paths and labels)
sample_image_paths = ["path/to/your/plant_image_1.jpg", "path/to/your/plant_image_2.jpg"]
sample_labels = [0, 1]

# 3. Instantiate your custom dataset
dataset = PlantDataset(
    image_paths=sample_image_paths, 
    labels=sample_labels, 
    transform=transform
)

# 4. Wrap it inside a PyTorch DataLoader
dataloader = DataLoader(
    dataset, 
    batch_size=2, 
    shuffle=True, 
    num_workers=0
)

# 5. Test fetching a batch to verify tensor shapes and types
images, labels = next(iter(dataloader))
print("Images tensor shape:", images.shape)  # Expected: torch.Size([2, 3, 224, 224])
print("Labels tensor:", labels)              # Expected: tensor([0, 1]) (or similar)

In [ ]:
import os
from typing import List, Tuple

def get_paths_and_labels(base_dir: str) -> Tuple[List[str], List[int], List[str]]:
    """
    Scans a directory structured as base_dir/class_name/image.jpg
    and extracts file paths and integer class labels.
    """
    image_paths = []
    labels = []
    
    # Sort class folders alphabetically to ensure consistent label mapping
    classes = sorted([d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))])
    class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
    
    for cls_name, class_idx in class_to_idx.items():
        cls_folder = os.path.join(base_dir, cls_name)
        for root, _, files in os.walk(cls_folder):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.webp')):
                    image_paths.append(os.path.join(root, file))
                    labels.append(class_idx)
                    
    return image_paths, labels, classes

In [ ]:
# 1. Extract paths and labels for each split directory
train_paths, train_labels, class_names = get_paths_and_labels("path/to/train")
val_paths, val_labels, _ = get_paths_and_labels("path/to/val")
test_paths, test_labels, _ = get_paths_and_labels("path/to/test")

print(f"Found {len(train_paths)} training images across {len(class_names)} classes: {class_names}")

# 2. Pass them directly into your PlantDataset
train_dataset = PlantDataset(image_paths=train_paths, labels=train_labels, transform=transform)
val_dataset = PlantDataset(image_paths=val_paths, labels=val_labels, transform=transform)

# 3. Create your DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
from collections import Counter
from ../src/data/dataset import get_dataloaders
train_loader, val_loader, test_loader, classes = get_dataloaders()
label_counts = Counter(train_loader.dataset.labels)
for class_idx, count in sorted(label_counts.items(), key=lambda x: x[1])[:15]:
    print(f"{classes[class_idx]:50s} {count}")